In [3]:
!pip install scikit-learn sqlalchemy pymysql pandas numpy

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sqlalchemy import create_engine

In [5]:
df = pd.read_csv('store_sales.csv')

In [6]:
print("Dataset shape : ", df.shape)


Dataset shape :  (7300, 5)


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7300 entries, 0 to 7299
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   date     7300 non-null   str    
 1   store    7300 non-null   int64  
 2   sales    7300 non-null   float64
 3   promo    7300 non-null   int64  
 4   holiday  7300 non-null   int64  
dtypes: float64(1), int64(3), str(1)
memory usage: 356.6 KB


In [8]:
df.head()

,date,store,sales,promo,holiday
0,2022-01-01,1,184.78,0,0
1,2022-01-02,1,192.62,0,0
2,2022-01-03,1,212.68,0,0
3,2022-01-04,1,249.58,1,0
4,2022-01-05,1,223.50,0,0


In [9]:
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

In [10]:
#Extract useful info from the date structure

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek

df[['date', 'year', 'month', 'day', 'dayofweek', 'sales', 'store']].head()

,date,year,month,day,dayofweek,sales,store
0,2022-01-01,2022,1,1,5,184.78,1
1,2022-01-02,2022,1,2,6,192.62,1
2,2022-01-03,2022,1,3,0,212.68,1
3,2022-01-04,2022,1,4,1,249.58,1
4,2022-01-05,2022,1,5,2,223.50,1


In [11]:
# 1. Define inputs (X) and target (y)

X = df[['store', 'year', 'month', 'day', 'dayofweek']]
y = df['sales']

# 2. Split data: 80% to train the model, 20% to test its accuracy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Create and train the model (Ridge Regression)
model = Ridge()
model.fit(X_train, y_train)  # 'fit' means: learn the patterns from the training data!

# 4. Check model accuracy score (R^2 score: closer to 1.0 is better)
print("Training Accuracy:", round(model.score(X_train, y_train), 4))
print("Testing Accuracy:", round(model.score(X_test, y_test), 4))

# 5. Generate predictions for all rows in your dataset
df['predicted_sales'] = model.predict(X)

# 6. Look at actual sales vs predicted sales side-by-side
df[['date', 'store', 'sales', 'predicted_sales']].head() 

Training Accuracy: 0.2963
Testing Accuracy: 0.3078


,date,store,sales,predicted_sales
0,2022-01-01,1,184.78,198.510173
1,2022-01-02,1,192.62,193.567804
2,2022-01-03,1,212.68,223.650587
3,2022-01-04,1,249.58,218.708218
4,2022-01-05,1,223.50,213.765849


In [15]:

# 1. Define business parameters
SAFETY_STOCK = 15
ASSUMED_CURRENT_STOCK = 60  # Simulated shelf stock

# 2. Define the Prescriptive Logic function
def decide_reorder_action(row):
    # Calculate total inventory needed to cover predicted demand + buffer
    needed_units = row['predicted_sales'] + SAFETY_STOCK
    
    # Check if demand exceeds what we currently have on the shelf
    if needed_units > ASSUMED_CURRENT_STOCK:
        # Calculate exact units to order (rounded up to nearest whole item)
        units_to_order = int(np.ceil(needed_units - ASSUMED_CURRENT_STOCK))
        return f"REORDER: {units_to_order} units"
    else:
        return "STOCK ADEQUATE: No Order Needed"

# 3. Apply the rule across all rows in your dataset
df['prescriptive_action'] = df.apply(decide_reorder_action, axis=1)

# 4. Inspect your predictions alongside the prescribed action
df[['date', 'store', 'sales', 'predicted_sales', 'prescriptive_action']].head(20)

,date,store,sales,predicted_sales,prescriptive_action
0,2022-01-01,1,184.78,198.510173,REORDER: 154 units
1,2022-01-02,1,192.62,193.567804,REORDER: 149 units
2,2022-01-03,1,212.68,223.650587,REORDER: 179 units
3,2022-01-04,1,249.58,218.708218,REORDER: 174 units
4,2022-01-05,1,223.50,213.765849,REORDER: 169 units
5,2022-01-06,1,221.79,208.823480,REORDER: 164 units
6,2022-01-07,1,200.43,203.881111,REORDER: 159 units
7,2022-01-08,1,213.47,198.938741,REORDER: 154 units
8,2022-01-09,1,222.44,193.996372,REORDER: 149 units
9,2022-01-10,1,203.10,224.079155,REORDER: 180 units


In [16]:
stock_adequate = df.loc[df["prescriptive_action"].str.contains("STOCK ADEQUATE", case=False, na=False)] 

In [18]:
print(stock_adequate)

Empty DataFrame
Columns: [date, store, sales, promo, holiday, year, month, day, dayofweek, predicted_sales, prescriptive_action]
Index: []
